In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.helperFunctions import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Update projected starting lineups

In [2]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 12 teams with confirmed lineups


### Load Model

### Load Player Data and Bookmaker Data

In [2]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Tyrese Maxey,Over,32.5,-137,2025-12-03,2025-12-02T23:15:07Z
1,PrizePicks,player_points,Tyrese Maxey,Under,32.5,-137,2025-12-03,2025-12-02T23:15:07Z
2,PrizePicks,player_points,C.J. McCollum,Over,19.5,-137,2025-12-03,2025-12-02T23:15:07Z
3,PrizePicks,player_points,C.J. McCollum,Under,19.5,-137,2025-12-03,2025-12-02T23:15:07Z
4,PrizePicks,player_points,Quentin Grimes,Over,17.5,-137,2025-12-03,2025-12-02T23:15:07Z


In [3]:
from PRODUCTION.featureEngine.feature_engine import FeatureEngine

# Initialize FeatureEngine with NGBOOST model for points prediction
engine = FeatureEngine({
    "min_model": "../MODELS/SAVED_MODELS/min_model.pkl",
    "usg_model": "../MODELS/SAVED_MODELS/usg_model.pkl",
    "ngboost_model_paths": {
        "mean_model": "../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl",
        "variance_model": "../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl",
        "calibration_factor": "../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl",
        "calibration_params": "../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_PARAMS_PRODUCTION.pkl",  # Add this
        "features": "../MODELS/SAVED_MODELS/pts_features.pkl"
    }
})

# Test prediction for a single player
result = engine.project_player(
    player_name="Draymond Green",
    data=s26,
    date="2025-11-30",
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

print("Prediction Result:")
print(f"  Predicted Minutes: {result['predicted_minutes']:.2f}")
print(f"  Predicted Usage: {result['predicted_usage']:.3f}")
print(f"  Predicted Points: {result['predicted_points']:.2f}")
print(f"\nFull result: {result}")

Prediction Result:
  Predicted Minutes: 31.20
  Predicted Usage: 0.185
  Predicted Points: 16.73

Full result: {'predicted_minutes': 31.19892692565918, 'predicted_usage': 0.18489530682563782, 'predicted_points': 16.731315919390074}


In [4]:
# In your LIVE.ipynb notebook
from scipy.stats import truncnorm

pred_data = get_cached_prediction_v2(
    player_name="Draymond Green",
    data=s26,
    engine=engine,
    current_date="2025-11-30",
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

mu = pred_data['prediction']
sigma = pred_data['sigma']
variance = pred_data['variance']

# Calculate 95% CI using truncated normal (can't go below 0)
lower_bound = 0
upper_bound = 50  # Reasonable max for any player

# Convert to standardized bounds
a = (lower_bound - mu) / sigma
b = (upper_bound - mu) / sigma

# Create truncated distribution
dist = truncnorm(a, b, loc=mu, scale=sigma)

# Get 95% confidence interval (2.5% and 97.5% percentiles)
ci_lower = dist.ppf(0.025)
ci_upper = dist.ppf(0.975)

print(f"Predicted Points (mu): {mu:.2f}")
print(f"Sigma (std dev): {sigma:.2f}")
print(f"Variance: {variance:.2f}")
print(f"95% Confidence Interval: [{ci_lower:.1f}, {ci_upper:.1f}]")

Predicted Points (mu): 16.73
Sigma (std dev): 6.80
Variance: 46.19
95% Confidence Interval: [4.1, 30.1]


## Top EVs for 2 leg bets

### Underdog picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

Pre-computing predictions for 56 players...
Processing 52 players...
Generated 1231 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
127,VJ Edgecombe,Derik Queen,12.5,13.5,-122,-105,20.77,22.80,0.904,0.890,over,over,1,136.42,0.682,Med,High
72,Tyrese Maxey,Jaden McDaniels,32.5,13.5,-109,-120,26.68,20.19,0.889,0.844,under,over,1,120.63,0.603,Med,High
45,Paul George,Jaylin Williams,15.5,4.5,-115,-137,21.84,8.97,0.832,0.833,over,over,1,103.83,0.519,High,Low
993,Donte DiVincenzo,Jalen Williams,13.5,18.5,-110,-108,19.30,24.34,0.828,0.819,over,over,1,99.49,0.497,Med,Med
323,Kyshawn George,Cason Wallace,15.5,7.5,105,100,21.72,12.89,0.766,0.799,over,over,1,80.01,0.400,High,High


### Prizepicks picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24)]

prizepicksPairs = calculate2LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 78 players...
Processing 73 players...
Generated 2433 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
365,VJ Edgecombe,Derik Queen,12.5,13.5,-122,-105,20.77,22.80,over,over,0.904,0.890,0.7881,0.368,0.390,0.519,136.42,0.682,1,5.97,7.21,Med,High,"(9.1, 32.5)","(8.7, 36.9)",0.05,0,136.4
1727,Jeremy Sochan,Draymond Green,5.5,9.5,100,-102,11.75,17.37,over,over,0.875,0.866,0.7425,0.388,0.373,0.501,122.76,0.614,1,5.22,6.80,Med,High,"(1.5, 22.0)","(4.1, 30.7)",0.05,0,122.8
1959,Jaden McDaniels,Jaylin Williams,13.5,4.5,-120,-137,20.19,8.97,over,over,0.844,0.833,0.6890,0.312,0.269,0.388,106.70,0.533,1,6.13,4.43,High,Low,"(8.2, 32.2)","(0.3, 17.6)",0.05,0,106.7
100,Paul George,Donte DiVincenzo,15.5,13.5,-115,-110,21.84,19.30,over,over,0.832,0.828,0.6755,0.310,0.317,0.407,102.65,0.513,1,6.07,5.60,High,Med,"(9.9, 33.7)","(8.3, 30.3)",0.05,0,102.6
2146,Mike Conley,Jalen Williams,4.5,18.5,104,-108,8.28,24.34,over,over,0.805,0.819,0.6461,0.326,0.313,0.403,93.84,0.469,0,4.12,5.85,Low,Med,"(0.2, 16.3)","(12.9, 35.8)",0.05,0,93.8
2358,Jordan Walsh,Cason Wallace,5.5,7.5,-124,100,9.80,12.89,over,over,0.800,0.799,0.6269,0.260,0.312,0.363,88.08,0.440,1,4.86,6.22,Low,High,"(0.3, 19.3)","(0.7, 25.1)",0.05,0,88.1
149,Kyshawn George,Devin Vassell,15.0,17.5,-137,106,21.72,24.34,over,over,0.785,0.794,0.6111,0.221,0.321,0.343,83.33,0.417,1,7.96,7.73,High,High,"(6.1, 37.3)","(9.2, 39.5)",0.05,0,83.3
1829,Saddiq Bey,Quinten Post,15.5,8.0,-110,-137,21.39,12.54,over,over,0.763,0.765,0.5720,0.252,0.201,0.283,71.60,0.358,1,7.60,5.95,High,Med,"(6.5, 36.3)","(0.9, 24.2)",0.05,0,71.6
1623,Luke Kornet,Jordan Hawkins,7.5,5.5,100,-122,11.38,8.79,over,over,0.762,0.760,0.5675,0.274,0.224,0.305,70.25,0.351,0,4.92,4.21,Low,Low,"(1.7, 21.0)","(0.5, 17.0)",0.05,0,70.3
236,Jared McCain,Naz Reid,14.5,13.5,-105,100,11.76,18.13,under,over,0.757,0.755,0.5606,0.258,0.268,0.315,68.19,0.341,0,4.62,5.99,Low,Med,"(2.7, 20.8)","(6.4, 29.9)",0.05,0,68.2


## 3 leg parlay

### Underdog picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') ]

underdogTrios = calculate3LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 56 players...
Processing 52 players...
Generated 17538 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
3023,VJ Edgecombe,Derik Queen,Jaylin Williams,12.5,13.5,4.5,20.77,22.80,8.97,0.904,0.890,0.833,over,over,over,1,261.72,0.523,Med,High,Low
1862,Tyrese Maxey,Jaden McDaniels,Jalen Williams,32.5,13.5,18.5,26.68,20.19,24.34,0.889,0.844,0.819,under,over,over,1,232.02,0.464,Med,High,Med
804,Paul George,Donte DiVincenzo,Cason Wallace,15.5,13.5,7.5,21.84,19.30,12.89,0.832,0.828,0.799,over,over,over,1,197.51,0.395,High,Med,High
6444,Kyshawn George,Julian Champagnie,Saddiq Bey,15.5,11.5,15.5,21.72,15.41,21.39,0.766,0.690,0.763,over,over,over,0,117.77,0.236,High,High,High
5809,Quentin Grimes,Naz Reid,Jaylen Brown,17.5,13.5,27.5,15.14,18.13,31.24,0.698,0.755,0.681,under,over,over,0,94.04,0.188,Med,Med,High


### Prizepicks picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

triosPrizepicks = calculate3LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 75 players...
Processing 70 players...
Generated 43098 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
10469,VJ Edgecombe,Jeremy Sochan,Derik Queen,12.5,5.5,13.5,20.77,11.75,22.80,0.904,0.875,0.890,over,over,over,1,280.15,0.560,Med,Med,High
3425,Paul George,Jaden McDaniels,Draymond Green,15.5,13.5,9.5,21.84,20.19,17.37,0.832,0.844,0.866,over,over,over,1,228.33,0.457,High,High,High
40164,Donte DiVincenzo,Jordan Walsh,Jalen Williams,13.5,5.5,18.5,19.30,9.80,24.34,0.828,0.800,0.819,over,over,over,1,193.25,0.386,Med,Low,Med
4580,Kyshawn George,Devin Vassell,Cason Wallace,15.0,17.5,7.5,21.72,24.34,12.89,0.785,0.794,0.799,over,over,over,1,169.15,0.338,High,High,High
36392,Luke Kornet,Saddiq Bey,Quinten Post,7.5,15.5,8.0,11.38,21.39,12.54,0.762,0.763,0.765,over,over,over,0,140.10,0.280,Low,High,Med
